In [1]:
import html
import json
import re
import sys
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

REPO_ROOT = Path.cwd().parent
DB_PATH = REPO_ROOT / "database" / "CSCAF_public.db"
OUTPUT_FOLDER = REPO_ROOT / "results" / "output"
FIGURE_FOLDER = REPO_ROOT / "results" / "figures"
CYBERHEAD_FILE = REPO_ROOT / "raw_data" / "supplementary" / "cyberhead_programmes.json"
SELECTION = REPO_ROOT / "raw_data" / "supplementary" / "selection.json"
MIN_ADS, TOP = 10, 3
VARIANTS = ("overview_learn_takeaways", "overview")
ADVERTISERS = json.loads((REPO_ROOT / "raw_data" / "supplementary" / "advertisers.json").read_text(encoding="utf-8"))
ADVERTISER = ADVERTISERS["cases"]
CASES = [("Sweden", ADVERTISER["SE-1"]), ("Belgium", ADVERTISER["BE-1"]), ("Ireland", ADVERTISER["IE-2"]),
         ("Hungary", ADVERTISER["HU-2"])]
FIGURE_ORDER = [CASES[0], CASES[2], CASES[1], CASES[3]]
INTERNATIONAL_ORGANISATIONS = set(ADVERTISERS["international_organisations"])
CYBERHEAD_ID = {"at_ustp": "b19f270e03ef46fc8a7850fec77d46c9", "be_kuleuven": "a50f6cba494d4d28aee7336421379a71",
                "cy_ouc": "1d4bcd7f1e8d466d813339bc83b59441", "dk_aau": "5a8ff01336204efeaf05d9995f394751",
                "fi_jamk": "d594e73891a84ddb9fc84c5134466e5f", "fr_ubs": "8e5df4eca30e4e968756ad5ed7e97764",
                "de_hdbw": "efa71c32e5d446bb8a22fa1e733f80e4", "gr_ihu": "bcd4012bc3e94f1a9f04b3df5933ed64",
                "hu_nke": "f0566942b9bc42dda6c04361c08c6084", "ie_mtu": "f14965c0b12d4a669403872067cbe845",
                "it_unipi": "1de3f0a4136f436a9510a2eca08ae710", "lv_lu": "d444ce8ceae1464d81dc18dbc0910e2c",
                "mt_um": "777f8d316de6453dbea44d76c1171fef", "pl_prz": "e663f5cafc674eec9fcb634b17d5b833",
                "pt_ipleiria": "b9480583d048407883473c67cfd72321", "ro_utcn": "86c8659261b74455a577e9dfa9dff0d9",
                "sk_tuke": "8d105dc71aa240c49a63b50188d5345a", "si_fis": "95c6840d25504be1851b32bbe5913dd4",
                "es_uah": "91bd9b7fab0642a7bde866d2bf457ee7", "se_kth": "723abd49b70b453b882fa4f96d773cbb"}
INTERMEDIARIES = ADVERTISERS["intermediaries"]
ROLE_PATTERNS = [("CISO", r"Chief Information Security Officer|CISO"), ("IncidentResponder", r"Incident Responder"),
                 ("LegalCompliance", r"Legal, Policy|Compliance Officer"), ("ThreatIntel", r"Threat Intelligence"),
                 ("Architect", r"Architect"), ("Auditor", r"Auditor"), ("Educator", r"Educator"),
                 ("Implementer", r"Implementer"), ("Researcher", r"Researcher"), ("RiskManager", r"Risk Manager"),
                 ("Forensics", r"Forensics? Investigator"), ("PenTester", r"Penetration Tester")]
ISO = {"Austria": "AT", "Belgium": "BE", "Bulgaria": "BG", "Croatia": "HR", "Cyprus": "CY", "Czechia": "CZ",
       "Denmark": "DK", "Estonia": "EE", "Finland": "FI", "France": "FR", "Germany": "DE", "Greece": "GR",
       "Hungary": "HU", "Ireland": "IE", "Italy": "IT", "Latvia": "LV", "Lithuania": "LT", "Luxembourg": "LU",
       "Malta": "MT", "Netherlands": "NL", "Poland": "PL", "Portugal": "PT", "Romania": "RO", "Slovakia": "SK",
       "Slovenia": "SI", "Spain": "ES", "Sweden": "SE"}
FIGURE_UNIVERSITY = {"se_kth": "KTH", "be_kuleuven": "KU Leuven", "ie_mtu": "MTU", "hu_nke": "Ludovika"}

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

assert DB_PATH.exists(), "run the notebooks of data_load first"

from cscaf.figures import EmployerPairsFigure
from cscaf.measures import (AdvertisementDemand, CourseAddition, CurriculumCoverage, DemandWeightedGap, KnownGroups,
                            RoleMapping)
from cscaf.storage import Storage
from cscaf.study import Study

study = Study.from_config()
TAU = study.tau
ENDS = study.ends
MIN_SHARE = study.section("characteristic_statements")["minimum_share"]
KEYS = study.curricula
TITLE_KEYWORDS = study.section("known_groups")["advertisement_titles"]
RUN = study.run_id
storage = Storage.of(study)
print("database :", DB_PATH.relative_to(REPO_ROOT))
print("run      :", RUN)
print("tau      :", TAU, " ends:", ENDS, " message share:", MIN_SHARE)
print("rule     :", f"top {TOP} advertisers per country, at least {MIN_ADS} advertisements, curriculum in CyberHEAD")

database : database\CSCAF_public.db
run      : sat-12l-sm_all-mpnet-base-v2_414c4542
tau      : 0.585  ends: (0.58, 0.595)  message share: 0.2
rule     : top 3 advertisers per country, at least 10 advertisements, curriculum in CyberHEAD


In [2]:
mapping = RoleMapping.read(storage)
per_role = {role: set(s) for role, s in mapping.statements_per_role().items()}
roles = sorted(per_role)
role_names = {e["role"]: e["role_name"] for entries in mapping.values() for e in entries}
n_roles_of = Counter(s for statements in per_role.values() for s in statements)
statement_text = dict(storage.framework.statement_texts(RUN))
programme_of = {country: key for key, country in storage.curricula.country_names()}
ads = storage.advertisements.listing()
title_of = {a[0]: a[3] for a in ads}
rows_of, sentence_of = defaultdict(list), {}
for ad, statement, score, sentence_id in storage.similarities.advertisement_best_match_rows(RUN):
    rows_of[ad].append((ad, statement, score))
    sentence_of[(ad, statement)] = sentence_id
all_ads = [a[0] for a in ads]


def coverage_at(threshold):
    moved = study.at_tau(threshold)
    result = CurriculumCoverage(moved, storage).statements(KEYS)
    return moved, {key: {s: v["coverage"] for s, v in result[key]["items"].items()} for key in KEYS}


def demand_of(ad_ids, moved):
    result = AdvertisementDemand(moved, storage).of_rows([r for a in ad_ids for r in rows_of[a]], len(ad_ids))
    return {s: v["binary_share"] for s, v in result["items"].items()}


STUDY_AT, COVERAGE_AT = {}, {}
for threshold in (TAU,) + ENDS:
    STUDY_AT[threshold], COVERAGE_AT[threshold] = coverage_at(threshold)
cyberhead = {r["ID"]: r["fields"] for r in
             json.loads(CYBERHEAD_FILE.read_text(encoding="utf-8"))["data"]["programmes"]["all_programmes"]["results"]}


def description_of(key):
    return re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ",
                                      html.unescape(cyberhead[CYBERHEAD_ID[key]]["programme_description"] or "")))


def declared_roles(key):
    text, found = description_of(key), set()
    for m in re.finditer(r"ECSF|Skills Framework", text):
        start = max(text.rfind(". ", 0, m.start()), 0)
        end = text.find(". ", m.end())
        window = text[start: (end if end > 0 else len(text))]
        if window.rstrip().endswith(":") or re.search(r"roles?\s*:", window):
            window = text[start: m.end() + 300]
        found |= {role for role, pattern in ROLE_PATTERNS if re.search(pattern, window)}
    return sorted(found)


def mentioned_roles(key):
    text = description_of(key)
    return sorted({role for role, pattern in ROLE_PATTERNS if re.search(pattern, text)} - set(declared_roles(key)))


def code_of(pair):
    return f"{ISO[pair['country']]}-{pair['rank']}"


print(f"{len(all_ads)} advertisements, {len(KEYS)} curricula, {len(CYBERHEAD_ID)} in CyberHEAD")

5136 advertisements, 27 curricula, 20 in CyberHEAD


In [3]:
per_country = Counter(a[2] for a in ads)
by_pair = defaultdict(list)
for ad, company, country, _ in ads:
    if company:
        by_pair[(country, company)].append(ad)
candidates = []
for country in sorted(per_country):
    top = sorted(((c, len(v)) for (k, c), v in by_pair.items() if k == country), key=lambda x: (-x[1], x[0]))[:TOP]
    for rank, (company, n) in enumerate(top, 1):
        candidates.append({"country": country, "company": company, "ads": n, "rank": rank,
                           "country_ads": per_country[country], "programme": programme_of.get(country),
                           "in_cyberhead": programme_of.get(country) in CYBERHEAD_ID,
                           "kind": INTERMEDIARIES.get(company, "employer")})
qualifying = [c for c in candidates if c["ads"] >= MIN_ADS and c["in_cyberhead"]]
print(f"top-{TOP} advertisers {len(candidates)}; with at least {MIN_ADS}: {sum(c['ads'] >= MIN_ADS for c in candidates)}; "
      f"curriculum in CyberHEAD: {len(qualifying)} in {len({c['country'] for c in qualifying})} countries "
      f"({sum(c['kind'] == 'employer' for c in qualifying)} employers, "
      f"{sum(c['kind'] != 'employer' for c in qualifying)} intermediaries)")

top-3 advertisers 81; with at least 10: 40; curriculum in CyberHEAD: 31 in 17 countries (22 employers, 9 intermediaries)


In [4]:
def ads_of(pair):
    if pair["company"] in INTERNATIONAL_ORGANISATIONS:
        return [a for (k, c), v in by_pair.items() if c == pair["company"] for a in v]
    return by_pair[(pair["country"], pair["company"])]


def analyse(pair, threshold):
    moved, cov_all = STUDY_AT[threshold], COVERAGE_AT[threshold]
    ad_ids = ads_of(pair)
    rest = [a for a in all_ads if a not in set(ad_ids)]
    b_set, b_rest = demand_of(ad_ids, moved), demand_of(rest, moved)
    lift = AdvertisementDemand.lift(b_set, b_rest, {role: per_role[role] for role in roles})
    by_lift = sorted(roles, key=lambda r: -lift[r])
    titled = Counter(role for a in ad_ids for role, kw in TITLE_KEYWORDS.items() if KnownGroups.matches(title_of[a], kw))
    titled_role = None
    if titled:
        role, count = max(titled.items(), key=lambda x: (x[1], -roles.index(x[0])))
        if count >= MIN_SHARE * len(ad_ids):
            titled_role = role
    role = titled_role or by_lift[0]
    key = pair["programme"]
    cov = cov_all[key]
    c_role = {r: np.mean([cov.get(s, 0.0) for s in per_role[r]]) for r in roles}
    place = 1 + sum(v > c_role[role] for v in c_role.values())
    asked = [s for s in per_role[role] if b_set.get(s, 0.0) >= MIN_SHARE]
    message = sorted([s for s in asked if not cov.get(s, 0.0)], key=lambda s: (n_roles_of[s] > 1, -b_set.get(s, 0.0)))
    entries = []
    for s in message:
        best_ad = max(ad_ids, key=lambda a: max((sc for _, st, sc in rows_of[a] if st == s), default=-1))
        sentence = storage.advertisements.sentence(sentence_of.get((best_ad, s)))
        entries.append({"statement_id": s, "statement": statement_text[s], "unique": n_roles_of[s] == 1,
                        "share": b_set.get(s, 0.0), "programmes_covering": int(sum(cov_all[k].get(s, 0.0) for k in KEYS)),
                        "sentence": sentence})
    declared = declared_roles(key)
    gap = DemandWeightedGap.of_role(b_set, cov, per_role[role])
    return {**pair, "code": code_of(pair), "tau": threshold, "n": len(ad_ids), "titled_role": titled_role,
            "titled_counts": dict(titled), "role": role, "lift": {r: lift[r] for r in by_lift}, "lift_first": by_lift[0],
            "no_preference": lift[by_lift[0]] < 1, "declared": declared, "mentioned": mentioned_roles(key),
            "declared_role": role in declared, "mentioned_role": role in mentioned_roles(key),
            "coverage_role": float(c_role[role]), "place_of_role": int(place),
            "coverage_roles": {r: float(v) for r, v in c_role.items()}, "gap": gap,
            "asked": len(asked), "message": entries}


def message_type(r):
    if r["no_preference"] and not r["titled_role"]:
        return "no message"
    if not r["message"]:
        return "no gap"
    if r["declared_role"]:
        return "confirm"
    if r["mentioned_role"]:
        return "strengthen"
    return "mission"


pairs = [analyse(p, TAU) for p in qualifying]
for r in pairs:
    r["type"] = message_type(r)
at_ends = {}
for threshold in ENDS:
    at_ends[threshold] = [analyse(p, threshold) for p in qualifying]
    for r in at_ends[threshold]:
        r["type"] = message_type(r)
for r in pairs:
    print(f"{r['code']:6s} {r['kind'][:26]:26s} {r['n']:3d}  {r['role']:18s} {'T' if r['titled_role'] else 'L'} "
          f"{'declared' if r['declared_role'] else '-':8s} {r['coverage_role']:.3f} ({r['place_of_role']:2d}) "
          f"G {r['gap'] if r['gap'] is not None else float('nan'):.3f}  {len(r['message'])}  {r['type']}")
print()
for threshold in ENDS:
    for e, r in zip(at_ends[threshold], pairs):
        if e["role"] != r["role"] or e["type"] != r["type"] or len(e["message"]) != len(r["message"]):
            print(f"at {threshold:.3f} {r['code']:6s} role {r['role']} -> {e['role']}, type {r['type']} -> {e['type']}, "
                  f"message {len(r['message'])} -> {len(e['message'])}")

BE-1   employer                    32  ThreatIntel        L -        0.348 (11) G 0.573  4  mission
BE-2   agency                      12  ThreatIntel        L -        0.348 (11) G 0.443  4  mission
BE-3   employer                    12  Forensics          L -        0.444 ( 9) G 0.460  4  mission
DK-1   employer                    12  LegalCompliance    L -        0.846 ( 2) G 0.038  0  no gap
FI-1   employer                    10  Forensics          L -        0.500 ( 9) G 0.043  0  no gap
FR-1   employer                    10  Forensics          L -        0.278 ( 6) G 0.714  5  mission
FR-2   employer                    10  CISO               T -        0.333 ( 4) G 0.537  9  mission
GR-1   employer                    11  LegalCompliance    L -        0.615 ( 7) G 0.333  2  mission
HU-1   job board                   40  Educator           L -        0.882 ( 5) G 0.000  0  no message
HU-2   agency (RPO)                20  Auditor            L -        0.500 (12) G 0.339  3  mission

In [5]:
case_rows = {(c, k): next(p for p in pairs if p["country"] == c and p["company"] == k) for c, k in CASES}
for (country, company), r in case_rows.items():
    print(f"\n{r['code']} ({r['kind']}, {r['n']} advertisements) -> {r['programme']}: role {r['role']}, "
          f"titles {r['titled_counts'] or 'none'}, lift first {r['lift_first']} {r['lift'][r['lift_first']]:.2f}")
    print(f"   declared {r['declared']}; C {r['coverage_role']:.3f}, place {r['place_of_role']}; G {r['gap']:.3f}")
    for e in r["message"]:
        print(f"   [{'unique' if e['unique'] else 'shared'}] {e['share']:.3f}  {e['statement']}")
hu2 = case_rows[("Hungary", ADVERTISER["HU-2"])]
audit_sentence_ads = storage.advertisements.count_with_sentence("Audit experience.", ads_of(hu2))
print("\nHU-2, advertisements with the sentence 'Audit experience.':", audit_sentence_ads, "of", hu2["n"])


SE-1 (employer, 30 advertisements) -> se_kth: role Architect, titles {'Architect': 8}, lift first Architect 2.30
   declared ['Architect', 'Implementer', 'PenTester']; C 0.840, place 1; G 0.075
   [unique] 0.467  Security architecture reference models
   [shared] 0.233  Legal, regulatory and legislative compliance requirements, recommendations and best practices

BE-1 (employer, 32 advertisements) -> be_kuleuven: role ThreatIntel, titles {'Auditor': 1, 'ThreatIntel': 4}, lift first ThreatIntel 1.87
   declared ['Architect', 'CISO', 'RiskManager']; C 0.348, place 11; G 0.573
   [unique] 0.281  Cross-domain and border-domain knowledge related to cybersecurity
   [unique] 0.250  Cyber threat actors
   [shared] 0.625  Communicate, present and report to relevant stakeholders
   [shared] 0.562  Communicate, coordinate and cooperate with internal and external stakeholders

IE-2 (employer, 10 advertisements) -> ie_mtu: role PenTester, titles {'PenTester': 2}, lift first ThreatIntel 1.50
   de

In [6]:
course_scores, course_sentences, course_roles = {v: {} for v in VARIANTS}, {v: {} for v in VARIANTS}, {}
titles = {}
for code, variant, rf, score, _ in storage.similarities.training_item_scores(RUN):
    if variant in VARIANTS:
        course_scores[variant].setdefault(code, {})[rf] = score
for training in storage.trainings.trainings(RUN):
    if training["variant"] in VARIANTS:
        course_sentences[training["variant"]][training["code"]] = training["sentences"]
        course_roles[training["code"]] = training["roles"]
        titles[training["code"]] = training["title"]
single_role = {c: rs[0] for c, rs in course_roles.items() if len(rs) == 1}
chosen = json.loads(SELECTION.read_text(encoding="utf-8"))["chosen"]
eligible = [r for r in case_rows.values() if r["kind"] == "employer" and r["role"] in set(single_role.values())]
loop_case = max(eligible, key=lambda r: (len(r["message"]), r["gap"] or 0.0))
LOOP_ROLE, LOOP_KEY = loop_case["role"], loop_case["programme"]
message_ids = [e["statement_id"] for e in loop_case["message"]]
loop_ads = ads_of(loop_case)
loop_targets = sorted(c for c, r in single_role.items() if r == LOOP_ROLE)
loop_controls = sorted(entry["code"] for role, entry in chosen.items()
                       if entry and role != LOOP_ROLE and single_role.get(entry["code"]) == role)
target_items = per_role[LOOP_ROLE]
programme_scores = {rf: e["similarity"] for rf, e in
                    CurriculumCoverage(study, storage).statements([LOOP_KEY])[LOOP_KEY]["items"].items()
                    if e["similarity"] is not None}


def ceiling(code):
    shared = set().union(*[per_role[r] for r in course_roles[code]]) & target_items
    return len(shared) / len(target_items)


def loop_row(codes, threshold, variant):
    moved = STUDY_AT[threshold]
    merged = {s: max([programme_scores.get(s, -1.0)] + [course_scores[variant][c].get(s, -1.0) for c in codes])
              for s in statement_text}
    after = {s: 1.0 if v >= threshold else 0.0 for s, v in merged.items()}
    before = {s: 1.0 if programme_scores.get(s, -1.0) >= threshold else 0.0 for s in statement_text}
    b_set = demand_of(loop_ads, moved)
    crossed = [s for s in message_ids if after[s] > before[s]]
    return {"delta": sum(after[s] - before[s] for s in target_items) / len(target_items),
            "message_covered": len(crossed), "message_covered_ids": crossed,
            "gap_before": DemandWeightedGap.of_role(b_set, COVERAGE_AT[threshold][LOOP_KEY], target_items),
            "gap_after": DemandWeightedGap.of_role(b_set, after, target_items)}


for code in loop_targets + loop_controls:
    single = CourseAddition(STUDY_AT[TAU]).change(programme_scores, course_scores[VARIANTS[0]][code], mapping,
                                                  LOOP_KEY, code, course_sentences[VARIANTS[0]][code], message_ids)
    assert abs(single.delta_for(LOOP_ROLE) - loop_row([code], TAU, VARIANTS[0])["delta"]) < 1e-12
loop = {code: {"use": "target" if code in loop_targets else "control", "roles": course_roles[code],
               "ceiling": None if code in loop_targets else ceiling(code),
               **loop_row([code], TAU, VARIANTS[0]),
               "ends": {t: loop_row([code], t, VARIANTS[0]) for t in ENDS},
               "narrow": loop_row([code], TAU, VARIANTS[1])} for code in loop_targets + loop_controls}


def reduction(code):
    return loop[code]["gap_before"] - loop[code]["gap_after"]


first = max(loop_targets, key=reduction)
first_ids = set(loop[first]["message_covered_ids"])
second = max((c for c in loop_targets + loop_controls if c != first),
             key=lambda c: (len(set(loop[c]["message_covered_ids"]) - first_ids), reduction(c)))
combination = {"courses": [first, second], **loop_row([first, second], TAU, VARIANTS[0]),
               "ends": {t: loop_row([first, second], t, VARIANTS[0]) for t in ENDS},
               "narrow": loop_row([first, second], TAU, VARIANTS[1])}
coordination = next(s for s, t in statement_text.items()
                    if t == "Communicate, coordinate and cooperate with internal and external stakeholders")
best_course_score = max(scores.get(coordination, -1.0) for scores in course_scores[VARIANTS[0]].values())
print(f"loop: {loop_case['code']} -> {LOOP_KEY}, role {LOOP_ROLE}, message {len(message_ids)}; "
      f"targets {loop_targets}; controls {loop_controls}")
for code, row in loop.items():
    print(f"{code:7s} {row['use']:7s} {row['delta']:+.3f} {row['message_covered']}/{len(message_ids)} "
          f"G {row['gap_before']:.3f} -> {row['gap_after']:.3f}  ceiling {row['ceiling']}  "
          f"ends {[round(row['ends'][t]['delta'], 3) for t in ENDS]}  narrow {row['narrow']['delta']:+.3f}")
print(f"combination {first} + {second}: {combination['delta']:+.3f}, {combination['message_covered']}/{len(message_ids)}, "
      f"G {combination['gap_before']:.3f} -> {combination['gap_after']:.3f}")
print(f"best course similarity to the coordination statement: {best_course_score:.3f}")
for code, scores in sorted(course_scores[VARIANTS[0]].items()):
    if scores.get(coordination, -1.0) >= TAU:
        print(f"   covered by {code} ({', '.join(course_roles[code])}) at {scores[coordination]:.3f}")

loop: BE-1 -> be_kuleuven, role ThreatIntel, message 4; targets ['FOR578', 'FOR589', 'SEC503', 'SEC599']; controls ['FOR498', 'FOR518', 'LDR521', 'SEC403', 'SEC497', 'SEC501', 'SEC530', 'SEC588']
FOR578  target  +0.217 1/4 G 0.573 -> 0.378  ceiling None  ends [0.217, 0.174]  narrow +0.217
FOR589  target  +0.130 1/4 G 0.573 -> 0.448  ceiling None  ends [0.13, 0.13]  narrow +0.000
SEC503  target  +0.087 0/4 G 0.573 -> 0.503  ceiling None  ends [0.13, 0.087]  narrow +0.000
SEC599  target  +0.217 1/4 G 0.573 -> 0.364  ceiling None  ends [0.217, 0.174]  narrow +0.217
FOR498  control +0.000 0/4 G 0.573 -> 0.573  ceiling 0.21739130434782608  ends [0.0, 0.0]  narrow +0.000
FOR518  control +0.000 0/4 G 0.573 -> 0.573  ceiling 0.30434782608695654  ends [0.0, 0.0]  narrow +0.000
LDR521  control +0.000 0/4 G 0.573 -> 0.573  ceiling 0.08695652173913043  ends [0.0, 0.0]  narrow +0.000
SEC403  control +0.087 2/4 G 0.573 -> 0.371  ceiling 0.13043478260869565  ends [0.087, 0.087]  narrow +0.043
SEC497 

In [7]:
FIGURE_FOLDER.mkdir(parents=True, exist_ok=True)
panels = []
for country, company in FIGURE_ORDER:
    r = case_rows[(country, company)]
    panels.append({"coverage": {k: round(v, 3) for k, v in r["coverage_roles"].items()}, "role": r["role"], "declared": r["declared"],
                   "title": f"{r['code']} \u2192 {FIGURE_UNIVERSITY.get(r['programme'], r['programme'])}\n"
                            f"{r['n']} advertisements"
                            + (", all countries" if company in INTERNATIONAL_ORGANISATIONS else f", {country}")
                            + "\n" + (f"G = {r['gap']:.3f}" if r["gap"] is not None else "G undefined")})
figure = EmployerPairsFigure(panels, sorted(roles, key=lambda r: role_names[r]), role_names).draw(
    FIGURE_FOLDER / "figure_8_employer_pairs.png")
print("written to", figure.relative_to(REPO_ROOT))

written to results\figures\figure_8_employer_pairs.png


In [8]:
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
output = OUTPUT_FOLDER / "07_employer_demand.json"
by_code = {r["code"]: r for r in pairs}
derived = {
    "cases": [r["code"] for r in case_rows.values()],
    "single_title": {r["code"]: len({title_of[a] for a in ads_of(r)}) == 1 for r in pairs},
    "be1_in_belgium": sum(1 for a in ads_of(by_code["BE-1"])
                          if a in set(by_pair[("Belgium", by_code["BE-1"]["company"])])),
    "si3_countries": len({a[2] for a in ads if a[0] in set(ads_of(by_code["SI-3"]))}),
    "es1_locations": len(set(storage.advertisements.locations(ads_of(by_code["ES-1"])))),
    "es1_texts": len(set(storage.advertisements.descriptions(ads_of(by_code["ES-1"])))),
    "audit_sentence_ads": audit_sentence_ads,
    "best_loop_course_similarity": max(course_scores[VARIANTS[0]][c].get(coordination, -1.0)
                                       for c in loop_targets + loop_controls),
    "role_statements": len(target_items),
}
record = {
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "paper_section": "4.3",
    "run": RUN, "tau": TAU, "ends": list(ENDS),
    "rule": {"top_per_country": TOP, "minimum_ads": MIN_ADS, "message_share": MIN_SHARE,
             "international_organisations": sorted(INTERNATIONAL_ORGANISATIONS), "intermediaries": INTERMEDIARIES},
    "candidates": candidates, "pairs": pairs, "pairs_at_ends": {str(t): v for t, v in at_ends.items()},
    "loop": {"case": loop_case["code"], "curriculum": LOOP_KEY, "role": LOOP_ROLE, "message": message_ids,
             "targets": loop_targets, "controls": loop_controls,
             "courses": {c: {**v, "ends": {str(t): e for t, e in v["ends"].items()}} for c, v in loop.items()},
             "combination": {**combination, "ends": {str(t): e for t, e in combination["ends"].items()}}},
    "derived": derived,
    "figures": [str(figure.relative_to(REPO_ROOT))],
}
output.write_text(json.dumps(record, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print("written to", output.relative_to(REPO_ROOT))

written to results\output\07_employer_demand.json
